# Piso 3ºB · Prueba híbrida: voz y emoción de Qwen + acento de España de Chatterbox
1. **Qwen** crea una grabación de cada personaje con cada emoción (voz distinta + emoción de verdad).
2. **Chatterbox España** usa esa grabación como modelo y vuelve a decir la frase **con acento de España**, copiando la voz y la emoción.
3. Te saca cada audio **numerado**. Dime cuáles suenan a España y con emoción.

*Entorno de ejecución → GPU T4 → Ejecutar todo.* La celda 1 reinicia la sesión una vez (normal): vuelve a pulsar *Ejecutar todo*. Tarda unos 10-15 minutos la primera vez.

In [ ]:
# 1) Librerías de Chatterbox (la primera vez reinicia la sesión una vez)
import os, subprocess
if not os.path.exists('/content/.hibrido_listo'):
    subprocess.run("pip install -q resampy librosa s3tokenizer omegaconf resemble-perth silero-vad conformer safetensors diffusers==0.29.0 transformers==4.46.3", shell=True)
    open('/content/.hibrido_listo', 'w').write('1')
    print("Librerías instaladas. Reiniciando la sesión... vuelve a pulsar «Ejecutar todo».")
    os.kill(os.getpid(), 9)
print("Librerías listas.")

In [ ]:
# 2) Qwen en un entorno aparte (para que no choque con Chatterbox): crea las grabaciones de cada personaje y emoción
import os, json, subprocess
os.makedirs('/content/refs', exist_ok=True)
ESP = ("Español de España, acento castellano de Madrid: distinción de 'z' y 'c' (como 'th'), 's' fuerte, jota fuerte, ritmo peninsular. "
       "Nada de acento latinoamericano ni neutro. ")
VOCES = {
 "yo":     ESP + "Hombre de unos 22 años, narrador cercano, irónico y cansado, tono medio.",
 "amigo":  ESP + "Hombre joven de unos 22 años, voz relajada y bromista, algo grave, muy coloquial.",
 "amiga":  ESP + "Mujer joven de unos 22 años, voz clara y enérgica, lista y sarcástica.",
 "jefe":   ESP + "Hombre de unos 55 años, voz ronca y autoritaria, casero gruñón.",
 "abuela": ESP + "Mujer de unos 75 años, voz temblorosa y cascada, vecina cotilla pero cariñosa.",
 "madre":  ESP + "Mujer de unos 50 años, voz claramente femenina, cálida y algo agobiada, madre española que se preocupa por todo.",
}
EMO = {
 "neutro":   "Habla con naturalidad.",
 "cansado":  "Habla muy cansado, con voz lenta y apagada, suspirando.",
 "enfadado": "Habla enfadado, con voz fuerte, tensa y rápida.",
 "alegre":   "Habla alegre y emocionado, con energía y sonriendo.",
 "panico":   "Habla en pánico, muy rápido y agudo, casi gritando.",
}
SEM = {"yo": 11, "amigo": 21, "amiga": 31, "jefe": 41, "abuela": 51, "madre": 61}
REFTXT = "Pues mira, yo creo que esto es lo que hay, y no hay mucho más que hablar, la verdad."
trabajo = [{"k": k, "e": e, "seed": SEM[k], "instruct": VOCES[k] + " " + EMO[e], "text": REFTXT, "out": f"/content/refs/{k}__{e}.wav"} for k in VOCES for e in EMO]
json.dump(trabajo, open('/content/trabajo.json', 'w'))
open('/content/gen_refs.py', 'w').write('''
import json, random, numpy as np, torch, soundfile as sf
from qwen_tts import Qwen3TTSModel
m = Qwen3TTSModel.from_pretrained("Qwen/Qwen3-TTS-12Hz-1.7B-VoiceDesign", device_map="cuda:0", dtype=torch.float16, attn_implementation="sdpa")
for i, t in enumerate(json.load(open("/content/trabajo.json"))):
    random.seed(t["seed"]); np.random.seed(t["seed"]); torch.manual_seed(t["seed"]); torch.cuda.manual_seed_all(t["seed"])
    w, sr = m.generate_voice_design(text=t["text"], language="Spanish", instruct=t["instruct"])
    sf.write(t["out"], w[0], sr); print(i + 1, "/", 30, t["k"], t["e"], flush=True)
''')
if not os.path.exists('/content/qv/bin/python'):
    # uv crea un entorno aparte con Python 3.12 (el Colab actual trae 3.13, donde Qwen puede fallar)
    r0 = subprocess.run("pip install -q uv && uv venv --python 3.12 /content/qv && uv pip install -q --python /content/qv/bin/python qwen-tts soundfile", shell=True)
    if r0.returncode != 0 or not os.path.exists('/content/qv/bin/python'):
        raise SystemExit("ERROR instalando Qwen en el entorno aparte (código %s). Copia el mensaje de arriba." % r0.returncode)
r = subprocess.run("/content/qv/bin/python /content/gen_refs.py", shell=True)
print("Grabaciones de Qwen listas." if r.returncode == 0 else "ERROR en Qwen (código %s)" % r.returncode)

In [ ]:
# 3) Cargar Chatterbox España en la GPU
import sys, os, random, numpy as np, torch, re, subprocess
if not os.path.exists('/content/sp'):
    subprocess.run("git clone -q https://huggingface.co/spaces/ResembleAI/Chatterbox-Multilingual-TTS-es-es /content/sp", shell=True, check=True)
sys.path.insert(0, '/content/sp'); os.chdir('/content/sp')
from chatterbox.src.chatterbox.tts import ChatterboxTTS
model = ChatterboxTTS.from_pretrained("cuda"); SR = model.sr; print("Chatterbox cargado.", SR)
def seed(n):
    random.seed(n); np.random.seed(n); torch.manual_seed(n); torch.cuda.manual_seed_all(n)

In [ ]:
# 4) Chatterbox lee cada frase con acento de España, copiando la voz y la emoción de la grabación de Qwen
import soundfile as sf, shutil
from IPython.display import Audio, display, HTML
os.makedirs('/content/salida', exist_ok=True)
EXAG = {"neutro": .6, "cansado": .5, "enfadado": 1.0, "alegre": .9, "panico": 1.2}
CFG  = {"neutro": .5, "cansado": .3, "enfadado": .5, "alegre": .5, "panico": .6}
FRASE = {
 "yo":     "Son las ocho y ocho, y ya llego tarde otra vez.",
 "amigo":  "Tranquilo, tío, que sobra tiempo. ¿Quién ha dejado la sartén en el suelo?",
 "amiga":  "No pasa nada, es puntualidad. Pero la nevera ha hecho un ruido muy raro.",
 "jefe":   "Esto es un piso, no un hotel. Quiero el alquiler antes del lunes, ¿vale?",
 "abuela": "Ay, hijo, qué ruido hacéis. En mis tiempos esto no pasaba.",
 "madre":  "Cariño, ¿has comido algo? Que te veo muy delgado, de verdad.",
}
n = 0
for k in VOCES:
    for e in EMO:
        n += 1; seed(SEM[k])
        w = model.generate(FRASE[k], exaggeration=EXAG[e], temperature=0.8, cfg_weight=CFG[e], language_id="es", audio_prompt_path=f"/content/refs/{k}__{e}.wav")
        f = f"/content/salida/{n:02d}_{k}_{e}.wav"; sf.write(f, w.squeeze(0).cpu().numpy(), SR)
        display(HTML(f"<div style='font:700 28px sans-serif;margin-top:14px'>Nº {n:02d} · {k} · {e}</div>")); display(Audio(f))

In [ ]:
# 5) Descargar todo (audios finales y las grabaciones de Qwen)
!cd /content && zip -qr hibrido.zip salida refs
from google.colab import files; files.download("/content/hibrido.zip")